<div align="right"><i>Matías Torres Esteban<br>Enero, 2026</i></div>

# Razonando con LLMs Pequeños

El objetivo de esta notebook es experimentar con LLMs locales y de bajos recursos para resolver problemas sencillos de razonamiento de múltiples pasos. La idea es adquirir una primera práctica con el análisis cualitativo y cuantitativo del comportamiento de un LLM al ser desplegado en flujos de trabajo que requieren alta precisión y correctitud. Veremos que incluso para problemas que parecen sencillos, el manejo seguro de los LLMs tiene muchos matices que dificultan su programación y que requieren técnicas ingenieriles poco convencionales. Nuestra notebook está organizada de la siguiente manera: 

1. En la primera sección presentamos el problema general a resolver por los LLMs y mostramos algunas características de nuestro lote de tareas.
2. En la segunda sección definimos el código necesario para invocar un LLM y procesar sus trazas de razonamiento y su respuesta final a un problema matemático o lógico sencillo.
3. En la tercera sección realizamos un estudio exhaustivo del comportamiento de los LLMs Gemma3:4b y Llama3.1:8b al variar sus parámetros de invocación, tal como los ejemplos escritos en la instrucción de sistema o el parafraseo de la misma. También analizamos el rendimiento de los LLMs sobre el lote de tareas completo.
4. En la última sección presentamos algunas conclusiones de los experimentos realizados y brindamos consejos de cómo  manipular LLMs pequeños dentro de un sistema informático. 

Esta notebook hace énfasis tanto en la experimentación y análisis de  los LLMs como en su modelado dentro de un sistema de software. Ambos aspectos son muy novedosos en el área de informática y es un problema abierto construir sistemas informáticos robustos y confiables cuya componente principal sean estos modelos. 

## 1. La Tarea

Le pediremos a los LLMs que resuelvan problemas aritméticos y lógicos sencillos pero que tienen una respuesta numérica bien definida. Nuestra intención es que dado el enunciado de un problema matemático, el modelo escriba su razonamiento paso a paso de cómo resolverlo junto a su solución final. Queremos que las respuestas estén estructuradas de manera tal que las trazas de razonamiento se encuentren dentro de un par de etiquetas `<Reasoning>` y la respuesta final esté escrita dentro de etiquetas `<Answer>`. Por ejemplo, dado el siguiente enunciado:

* *Enunciado: ¿Si María tiene 3 bananas y Pedro tiene 2 manzanas y el doble de bananas de María, cuántas frutas tienen ambos en total?*

Una respuesta correcta y bien formateada sería: 

```xml
<Reasoning> 
    Si Pedro tiene el doble de bananas que Maria y María tiene 3 bananas, entonces Pedro
    tiene 3 * 2 = 6 bananas. Por lo tanto, la cantidad de frutas total es 3 + 2 + 6 = 11. 
</Reasoning>
<Answer>
    11
</Answer>
```

La generación de trazas o *cadenas de pensamiento* tiene muchas ventajas dentro de un sistema de preguntas y respuestas. Una de sus mayores virtudes es que permite al usuario validar o verificar las respuestas que genera un modelo a preguntas sensibles, que solicitan contenido factual y que requieren un alto grado de precisión. También fue descubierto que permitir que el modelo genere cadenas de pensamiento antes de brindar una respuesta definitiva a una consulta mejoraba su efectividad y precisión para resolver problemas. 

Con la ayuda de LLMs más grandes como GPT4 y Gemini sintetizamos un pequeño conjunto de problemas y sus soluciones. Estos problemas los almacenamos en un archivo `shots.json`. También generamos diferentes parafraseos de la siguiente instrucción de sistema:

* *You are a mathematician skilled in arithmetic problem solving. Solve the problem using accurate logic and calculations.*

Estos los almacenamos en el archivo `system.json` y nuestro objetivo con ellos es probar la robustez de los LLMs para seguir la misma instrucción pero escrita de diferentes modos. 

A continuación definimos las clases de nuestro dominio y cargamos el lote de tareas e instrucciones para invocar al modelo. 

In [1]:
from dataclasses import dataclass
import json

@dataclass 
class Shot:
    
    shot_id: int = 0
    statement: str = ''
    reasoning: str = ''
    answer: str = ''

@dataclass 
class SystemInstruction:
    
    instruction_id: int = 0
    text: str = ''

with open('../data/cot/system.json', 'r') as f:
    data = json.load(f)
    
    instructions = [SystemInstruction(**datum) 
                    for datum in data]

with open('../data/cot/shots.json', 'r') as f:
    data = json.load(f)
    
    shots = [Shot(**datum) 
             for datum in data]

Ahora definiremos la componente encargada de formatear las instrucciones de sistema y usuario.

In [2]:
class PromptFormatter:
    """
    Formats the user and system prompts for solving
    a reasoning task. 
    """

class PromptFormatter:
    """ 
    Formats the user and system prompts
    for solving a reasoning task. 
    """
    
    def __init__(self, statement_prompt, opening_reasoning_tag, closing_reasoning_tag, 
                 opening_answer_tag, closing_answer_tag):
        
        self._statement_prompt = statement_prompt
        self._opening_reasoning_tag = opening_reasoning_tag
        self._closing_reasoning_tag = closing_reasoning_tag
        self._opening_answer_tag = opening_answer_tag
        self._closing_answer_tag = closing_answer_tag

    def format_system_prompt(self, system_instruction: SystemInstruction, shot: Shot):
        """ Uses the one shot strategy """
        
        return (f"{system_instruction.text}\n"
                "Example:\n"
                f"{self._statement_prompt}: {shot.statement}\n"
                f"{self._opening_reasoning_tag}\n"
                f"{shot.reasoning}\n"
                f"{self._closing_reasoning_tag}\n"
                f"{self._opening_answer_tag}\n"
                f"{shot.answer}\n"
                f"{self._closing_answer_tag}")

    def format_user_prompt(self, shot: Shot):
        return (f"{self._statement_prompt}: {shot.statement}\n"
                f"{self._opening_reasoning_tag}\n")

Asi es como quedaría una instrucción de sistema:

In [3]:
formatter = PromptFormatter('Statement', '<Reasoning>', '</Reasoning>', '<Answer>', '</Answer>')
print(formatter.format_system_prompt(instructions[0], shots[0]))

You are a mathematician skilled in arithmetic problem solving. Solve the problem correctly.
- Use <Reasoning> to give a brief justification of the approach.
- Use <Answer> to present the final result.
Example:
Statement: If a baker makes 12 loaves of bread in the morning and 15 in the afternoon, how many loaves did they make in total?
<Reasoning>
To find the total, add the morning production to the afternoon production: 12 + 15 = 27.
</Reasoning>
<Answer>
27
</Answer>


Esta sería la instrucción de usuario:

In [4]:
print(formatter.format_user_prompt(shots[1]))

Statement: A library has 150 books. If 37 books are checked out, how many books remain on the shelves?
<Reasoning>



## 2. Análisis Sintáctico

En esta sección escribimos el código necesario para extraer el razonamiento y la respuesta final de un LLM a un problema lógico y haremos una pequeña demostración de su funcionamiento con los LLMs Gemma3:4b y Llama3.1:8b. 

La extracción de estas componentes estará a cargo de la clase `AnswerParser` que definimos a continuación:

In [5]:
from llms_kgs.llms import LLMInvocationData
from typing import Tuple
import re

class AnswerParser:
    
    """ 
    Extracts the reasoning and final answer components 
    from the LLM generated text to a reasoning task.
    """
    
    def __init__(self, opening_reasoning_tag: str, closing_reasoning_tag: str, 
                 opening_answer_tag: str, closing_answer_tag: str):
        
        self._opening_reasoning_tag = opening_reasoning_tag
        self._closing_reasoning_tag = closing_reasoning_tag
        self._opening_answer_tag = opening_answer_tag
        self._closing_answer_tag = closing_answer_tag

        self._reasoning_pattern = f'{opening_reasoning_tag}.*?{closing_reasoning_tag}'
        self._answer_pattern = f'{opening_answer_tag}.*?{closing_answer_tag}'

    def extract_answer_component(self, invocation: LLMInvocationData) -> str:
        
        search_result = re.search(self._answer_pattern, invocation.raw_answer, re.DOTALL)
        if not search_result:
            raise ValueError("Could not extract answer.")
        
        answer = search_result.group()
        return answer[len(self._opening_answer_tag): -len(self._closing_answer_tag)].strip()
        
    def extract_reasoning_component(self, invocation: LLMInvocationData) -> str:
        
        search_result = re.search(self._reasoning_pattern, invocation.raw_answer, re.DOTALL)
        if not search_result:
            raise ValueError("Could not extract reasoning.")

        reasoning = search_result.group()
        return reasoning[len(self._opening_reasoning_tag): -len(self._closing_reasoning_tag)].strip()

    def parse(self, invocation: LLMInvocationData) -> Tuple[str, str]: 

        return (self.extract_reasoning_component(invocation),
                self.extract_answer_component(invocation))
        

Invocamos los modelos Llama3.1:8b y Gemma3:4b con los prompts de sistema y usuario que acabamos de crear.

In [6]:
from llms_kgs.llms import Llama_31_8B, Gemma3_4B

llama = Llama_31_8B(temperature = 0.0)
gemma = Gemma3_4B(temperature = 0.0)

system = formatter.format_system_prompt(instructions[0], shots[0])
prompt = formatter.format_user_prompt(shots[1])
                                      
llama_invocation = llama.call(system=system, prompt=prompt)
gemma_invocation = gemma.call(system=system, prompt=prompt)

Las respuestas en bruto de Llama3.1:8b y Gemma3:4b son:

In [7]:
print("llama3.1:8b")
print(llama_invocation.raw_answer)
print("="*80)
print("gemma3:4b")
print(gemma_invocation.raw_answer)

llama3.1:8b
<Reasoning>
To find the number of books remaining on the shelves, subtract the number of books checked out from the total number of books in the library: 150 - 37 = 113.
</Reasoning>

<Answer>
113
</Answer>
gemma3:4b
<Reasoning>
To find the number of books remaining, subtract the number of books checked out from the total number of books: 150 - 37 = 113.
</Reasoning>
<Answer>
113
</Answer>


Estas son las componentes individuales para Llama3.1:8b extraídas con el analizador sintáctico:

In [8]:
parser = AnswerParser(
    '<Reasoning>', 
    '</Reasoning>', 
    '<Answer>', 
    '</Answer>')

reasoning, answer = parser.parse(llama_invocation)

print(f"Extracted Reasoning: {reasoning}")
print(f"Extracted Answer: {answer}")

Extracted Reasoning: To find the number of books remaining on the shelves, subtract the number of books checked out from the total number of books in the library: 150 - 37 = 113.
Extracted Answer: 113


## 3. Análisis Cuantitativo

En esta sección realizamos algunos experimentos y estudiamos la efectividad de los LLMs para llevar a cabo razonamientos sencillos bajo diferentes variaciones en su instrucción. Para ello, definimos las abstracciones necesarias para ejecutar y evaluar los experimentos y brindamos ejemplos de su uso. Nuestra intención es construir un marco de trabajo genérico que nos permita realizar otras investigaciones en el futuro.

A continuación definimos la clase `ExperimentReport`, la cual almacena los resultados más importantes de un experimento. Sus componentes son:

* `invocation`: Lista de todas las invocaciones realizadas al LLM.
* `wrongly_formatted_invocations`: Lista de invocaciones que generaron un error de análisis sintáctico.
* `incorrect_answers`: Lista de invocaciones que fueron procesadas exitosamente por el analizador sintáctico pero cuya respuesta final no es correcta.
* `distinct_reasonings`: Lista de los diferentes razonamientos generados por un LLM al ser invocado varias veces. 
* `accuracy`: Precisión del modelo al resolver un lote de tareas. 

In [9]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class ExperimentReport:
    """ Stores and categorizes the results of a robustness experiment. """
    
    invocations: List[LLMInvocationData] = field(default_factory=list)
    wrongly_formatted_invocations: List[LLMInvocationData] = field(default_factory=list)
    incorrect_answers: List[LLMInvocationData] = field(default_factory=list)
    distinct_reasonings: List[str] = field(default_factory=list)
    accuracy: float = 0.0 

    def summary(self) -> str:
        """ Returns a quick summary of the report. """
        
        total = len(self.invocations)
        return (f"--- Experiment Summary ---\n"
                f"Total Invocations:  {total}\n"
                f"Success (Accuracy): {self.accuracy:.2%}\n"
                f"Format Errors:      {len(self.wrongly_formatted_invocations)}\n"
                f"Incorrect Answers:  {len(self.incorrect_answers)}\n"
                f"--------------------------")

También definimos una clase abstracta `AbstractExperiment` y que representa el orquestrador lógico de un experimento. Su método principal `execute` recibe un LLM y devuelve un reporte con los resultados del experimento. 

In [10]:
from abc import ABC, abstractmethod
from llms_kgs.llms import LLMProtocol

class AbstractExperiment:

    @abstractmethod
    def execute(self, llm: LLMProtocol) -> ExperimentReport:
        """ Orchestrates the main logic to execute a experiment """
        pass 

### 3.1 Variación de Instrucción

Nuestro primer experimento consistirá en variar el parafraseo de la instrucción de sistema dejando fijos el ejemplo brindado y la tarea a resolver. Queremos evaluar si el LLM es capaz de generar la misma respuesta correcta a pesar de variar la redacción de su comportamiento general. 

A continuación presentamos dos redacciones distintas del mismo comportamiento esperado:

In [16]:
print(instructions[3].text)
print("="*80)
print(instructions[4].text)

Assume the role of a professional mathematician. Solve the given problem accurately.
- Use <Reasoning> for a high-level explanation.
- Use <Answer> for the final outcome.
You are trained in solving arithmetic word problems.
- Place a concise rationale in <Reasoning>.
- Place the final answer in <Answer>.


Definimos la clase encargada de ejecutar y evaluar el experimento para diferentes LLMs:

In [18]:
from typing import List, Dict, Optional

class InstructionVariationExperiment(AbstractExperiment):
    """
    Orchestrates experiments of instruction variation
    keeping the task and shot constant. 
    """

    def __init__(
        self,
        formatter: PromptFormatter,
        parser: AnswerParser,
        instructions: List[SystemInstruction],
        shot: Shot,
        task: Shot
    ):
        self._formatter = formatter
        self._parser = parser
        self._instructions = instructions
        self._shot = shot
        self._task = task
        
        self._user_prompt = self._formatter.format_user_prompt(task)

    def execute(self, llm: LLMProtocol) -> ExperimentReport:
        
        invocations = []
        for instruction in self._instructions:
            
            system_prompt = self._formatter.format_system_prompt(instruction, self._shot)
            invocation = llm.call(system=system_prompt, prompt=self._user_prompt)
            invocations.append(invocation)

        return self._fill_report(invocations)

    def _fill_report(self, invocations: LLMInvocationData) -> ExperimentReport: 
        """ Computes the main results of the experiment """
        
        report = ExperimentReport(invocations=invocations)
        
        correct_matches = 0
        for invocation in report.invocations:
            try:
            
                reasoning, answer = self._parser.parse(invocation)

                if str(answer).strip() == str(self._task.answer).strip():
                    correct_matches += 1
                else:
                    report.incorrect_answers.append(invocation)

                if reasoning not in report.distinct_reasonings:
                    report.distinct_reasonings.append(reasoning)
            
            except ValueError:
                report.wrongly_formatted_invocations.append(invocation)

        if report.invocations:
            report.accuracy = correct_matches / len(report.invocations)

        return report

A continuación ejecutamos el experimento e imprimimos el resumen de los resultados:

In [19]:
experiment = InstructionVariationExperiment(formatter, parser, instructions, shots[0], shots[1]) 

gemma_report = experiment.execute(gemma)
llama_report = experiment.execute(llama)

print('gemma3:4b')
print(gemma_report.summary())
print('='*80)
print('llama3.1:8b')
print(llama_report.summary())

gemma3:4b
--- Experiment Summary ---
Total Invocations:  30
Success (Accuracy): 93.33%
Format Errors:      1
Incorrect Answers:  1
--------------------------
llama3.1:8b
--- Experiment Summary ---
Total Invocations:  30
Success (Accuracy): 83.33%
Format Errors:      5
Incorrect Answers:  0
--------------------------


Perfecto! Ya tenemos nuestros primeros resultados. Observamos que ninguno de los dos modelos obtuvo una precisión perfecta para resolver la tarea dada. El modelo de Gemma3:4b tiene 1 error de formato y 1 error de respuesta, mientras que todas las equivocaciones de Llama3.1:8b son de formato. Examinaremos en las próximas celdas de código los errores en detalle.

Comenzamos por Llama3.1:8b e imprimimos sus errores de formato:

In [21]:
for x in llama_report.wrongly_formatted_invocations:
    print(x.raw_answer)
    print("="*80)

To find the number of books remaining on the shelves, subtract the number of books checked out from the total number of books in the library: 150 - 37 = 113.
</Reasoning>
<Answer>
113
</Answer>
To find the number of books remaining on the shelves, subtract the number of books checked out from the total number of books: 150 - 37 = 113.
</Reasoning>
To find the number of books remaining on the shelves, subtract the number of books checked out from the total number of books:

Total number of books = 150
Number of books checked out = 37

Remaining books = Total number of books - Number of books checked out
= 150 - 37
= 113

</Reasoning>

<Answer>
113
</Answer>
To find the number of books remaining on the shelves, subtract the number of books checked out from the total number of books: 150 - 37 = 113.
</Reasoning>

<Answer>
113
</Answer>
To find the number of books remaining on the shelves, subtract the number of books checked out from the total number of books:

150 (total books) - 37 (boo

Observamos que en todos estos casos falta la etiqueta de apertura `<Reasoning>`, pero las respuestas finales brindadas son las correctas. 

Ahora examinemos los errores de formato y de respuesta de Gemma3:4b:

In [25]:
print(gemma_report.wrongly_formatted_invocations[0].raw_answer)
print("="*80)
print(gemma_report.incorrect_answers[0].raw_answer)

To find the number of books remaining, subtract the number of books checked out from the total number of books: 150 - 37 = 113.
</Reasoning>
<Answer>
113
</Answer
<Reasoning>
To find the number of books remaining, we need to subtract the number of books checked out from the total number of books in the library. This is a straightforward subtraction problem.
</Reasoning>
<Answer>
150 - 37 = 113
113
</Answer>


Observamos que el error de formato es el mismo que el que sufrió Llama3:8b. Con respecto al error de respuesta, vemos que el modelo genera el valor correcto pero en un formato que no era el esperado - Se requería la cadena "113". 

Podemos realizar un análisis también de los diferentes razonamientos producidos para llegar a un resultado. Estos son las trazas de razonamiento recuperadas del modelo Gemma3:4b: 

In [27]:
for x in gemma_report.distinct_reasonings:
    print(f"{x}\n")

To find the number of books remaining, subtract the number of books checked out from the total number of books: 150 - 37 = 113.

To find the number of books remaining, we need to subtract the number of books checked out from the total number of books in the library. This is a straightforward subtraction problem.

To find the number of books remaining, we subtract the number of books checked out from the total number of books in the library: 150 - 37 = 113.

To find the number of books remaining, we need to subtract the number of books checked out from the total number of books in the library.
Calculation: 150 - 37 = 113

We need to subtract the number of books checked out from the total number of books in the library to find the number of books remaining. This is a simple subtraction problem: 150 - 37 = ?



Estos son los razonamientos producidos por Llama3.1:8b:

In [28]:
for x in llama_report.distinct_reasonings:
    print(f"{x}\n")

To find the number of books remaining on the shelves, subtract the number of books checked out from the total number of books in the library: 150 - 37 = 113.

To find the number of remaining books, subtract the number of checked-out books from the total number of books: 150 - 37 = 113.

To find the number of remaining books, subtract the number of books checked out from the total number of books: 150 - 37 = 113.

To find the number of books remaining on the shelves, subtract the number of books checked out from the total number of books: 150 - 37 = 113.



Es interesante que a pesar de tener 30 instrucciones de sistemas distintas, la cantidad de cadenas de pensamiento producidas varía entre 4 y 5. Esto es un buen resultado porque pareciera que el razonamiento generado para resolver un problema está débilmente condicionado por las pequeñas variaciones en la redacción del comportamiento del LLM. 

Finalmente imprimimos estadísticas de los tiempos de ejecución. Ambos modelos fueron desplegados en su completitud en una GPU local al momento de ser invocados: 

In [20]:
import numpy as np

def print_time_statistics(invocations: List[LLMInvocationData]):
    """
    Helper function that prints basic time
    statistics  of a sequence of LLM invocations. 
    """
    
    times = np.array([invocation.execution_time for invocation in invocations])
    print(f'Total: {np.sum(times)}')
    print(f'Average: {np.mean(times)}')
    print(f'Max: {np.max(times)}')
    print(f'Min: {np.min(times)}')

print('Llama3.1:8b')
print_time_statistics(llama_report.invocations)
print("="*80)
print('Gemma3:4b')
print_time_statistics(gemma_report.invocations)

Llama3.1:8b
Total: 176.29619097709656
Average: 5.8765396992365515
Max: 17.20921564102173
Min: 2.571070671081543
Gemma3:4b
Total: 134.65008544921875
Average: 4.488336181640625
Max: 21.152971982955933
Min: 2.2634665966033936


El modelo más pequeño Gemma3:4b tuvo los tiempos de ejecución más cortos. 

### 3.2 Variación de Shot

Nuestro segundo experimento consiste en variar el ejemplo de entrenamiento (shot) escrito en la instrucción de sistema. El procedimiento a seguir para obtener los resultados es muy similar al que utilizamos al variar el parafraseo del comportamiento del LLM. Debemos definir una clase coordinadora de la ejecución del experimento y la obtención del reporte para luego realizar una interpretación de los resultados.

La clase `ShotVariationExperiment` tiene una estructura muy similar a la clase `InstructionVariationExperiment` pero ahora variamos el ejemplo de entrenamiento antes que la instrucción:

In [12]:
from typing import List, Dict, Optional

class ShotVariationExperiment(AbstractExperiment):
    """
    Orchestrates experiments of shot variation
    keeping the task and instruction constant. 
    """

    def __init__(
        self,
        formatter: PromptFormatter,
        parser: AnswerParser,
        instruction: SystemInstruction,
        shots: List[Shot],
        task: Shot
    ):
        self._formatter = formatter
        self._parser = parser
        self._instruction = instruction
        self._shots = shots
        self._task = task
        
        self._user_prompt = self._formatter.format_user_prompt(task)

    def execute(self, llm: LLMProtocol) -> ExperimentReport:
        
        invocations = []
        for shot in self._shots:
            
            system_prompt = self._formatter.format_system_prompt(self._instruction, shot)
            invocation = llm.call(system=system_prompt, prompt=self._user_prompt)
            invocations.append(invocation)

        return self._fill_report(invocations)

    def _fill_report(self, invocations: LLMInvocationData) -> ExperimentReport: 
        """ Computes the main results of the experiment """
        
        report = ExperimentReport(invocations=invocations)
        
        correct_matches = 0
        for invocation in report.invocations:
            try:
            
                reasoning, answer = self._parser.parse(invocation)

                if str(answer).strip() == str(self._task.answer).strip():
                    correct_matches += 1
                else:
                    report.incorrect_answers.append(invocation)

                if reasoning not in report.distinct_reasonings:
                    report.distinct_reasonings.append(reasoning)
            
            except ValueError:
                report.wrongly_formatted_invocations.append(invocation)

        if report.invocations:
            report.accuracy = correct_matches / len(report.invocations)

        return report

Ejecutamos el experimento e imprimimos un resumen de los resultados:

In [13]:
experiment = ShotVariationExperiment(formatter, parser, instructions[0], shots, shots[1]) 

gemma_report = experiment.execute(gemma)
llama_report = experiment.execute(llama)

print('gemma3:4b')
print(gemma_report.summary())
print('='*80)
print('llama3.1:8b')
print(llama_report.summary())

gemma3:4b
--- Experiment Summary ---
Total Invocations:  30
Success (Accuracy): 46.67%
Format Errors:      15
Incorrect Answers:  1
--------------------------
llama3.1:8b
--- Experiment Summary ---
Total Invocations:  30
Success (Accuracy): 96.67%
Format Errors:      1
Incorrect Answers:  0
--------------------------


Muy interesante! El rendimiento de Llama3.1:8b mejoró sustancialmente mientras que el de Gemma3:4b empeoró. Analicemos ahora el error de formato de Llama3.1:8b:

In [14]:
print(llama_report.wrongly_formatted_invocations[0].raw_answer)

Subtract the number of books checked out from the initial total: 150 - 37 = 113.
</Reasoning>

<Answer>
113
</Answer>


Ocurrió el mismo error de formato donde el modelo no escribió la primera etiqueta `<Reasoning>`. Analicemos ahora los errores de formato de Gemma3:4b:

In [19]:
for x in gemma_report.wrongly_formatted_invocations:
    print(x.raw_answer)
    print("="*80)

<Reasoning>
To find the number of books remaining on the shelves, we need to subtract the number of books checked out from the total number of books in the library.
</Reasoning>
<Answer>
113
</Answer
<Reasoning>
Subtract the number of checked-out books from the total number of books to find the remaining books.
</Reasoning>
<Answer>
150 - 37 = 113

<Reasoning>
We need to subtract the number of books checked out from the total number of books in the library. This is a straightforward subtraction problem.
</Reasoning>
<Answer>
150 - 37 = 113
There are 113 books remaining on the shelves.
<Reasoning>
We need to subtract the number of books checked out from the total number of books in the library to find the number of books remaining.
</Reasoning>
<Answer>
113

<Reasoning>
Subtract the number of checked-out books from the total number of books: 150 - 37 = 113.
</Reasoning>
<Answer>
113

<Reasoning>
To find the number of books remaining, we need to subtract the number of books checked out f

Vemos que el modelo comete repetidamente el error de no escribir la última etiqueta `</Answer>` o la escribe de forma incorrecta. Esto sugiere que el formato con que el LLM genera su respuesta es sensible a los ejemplos de entrenamiento. A pesar de estos errores de formato, todas estas invocaciones contienen en esencia la respuesta correcta a la tarea. Quizás un postprocesamiento de la respuesta del modelo podría ayudar a recuperar su respuesta generada, o también podríamos escribir más ejemplos en su instrucción de sistema para alterar su comportamiento. 

El error de respuesta de Gemma3:4b es:

In [22]:
print(gemma_report.incorrect_answers[0].raw_answer)

<Reasoning>
To find the number of books remaining, subtract the number of books checked out from the total number of books.
</Reasoning>
<Answer>
150 - 37 = 113
</Answer>


De nuevo, el modelo genero la respuesta correcta pero no en el formato que esperabamos. Una futura normalización de la respuesta podría ayudarnos a extraer el número que deseamos. 

Imprimamos las cadenas de pensamiento producidas por Gemma3:4b:

In [24]:
for x in gemma_report.distinct_reasonings:
    print(f"{x}\n")

To find the number of books remaining, subtract the number of books checked out from the total number of books: 150 - 37 = 113.

Subtract the number of checked-out books from the total number of books in the library: 150 - 37 = 113.

We need to subtract the number of books checked out from the total number of books in the library to find the number of books remaining.

To find the number of books remaining, subtract the number of books checked out from the total number of books.

Subtract the number of checked-out books from the total number of books to find the remaining books.

Subtract the number of books checked out (37) from the total number of books (150): 150 - 37 = 113.

We need to subtract the number of books checked out from the total number of books to find the number of books remaining.

Subtract the number of checked-out books from the total number of books: 150 - 37 = 113.

We need to subtract the number of books checked out from the total number of books to find the numb

Estas son las cadenas de pensamiento producidas por Llama3.1:8b:

In [25]:
for x in llama_report.distinct_reasonings:
    print(f"{x}\n")

To find the number of books remaining on the shelves, subtract the number of books checked out from the total number of books in the library: 150 - 37 = 113.

To find the number of books remaining on the shelves, subtract the number of books checked out from the total number of books in the library.

To find the number of books remaining on the shelves, subtract the number of books checked out from the total number of books in the library (150 - 37).

Subtract the number of books checked out from the total number of books to find the remaining number of books.

To find the number of books remaining on the shelves, subtract the number of books that were checked out from the total number of books in the library: 150 - 37.

To find the number of books remaining on the shelves, subtract the number of books checked out from the total number of books in the library.
Total books = 150
Books checked out = 37
Remaining books = Total books - Books checked out
= 150 - 37
= 113

To find the number

Wow! El número de razonamientos producidos por ambos modelos aumentó considerablemente respecto al experimento anterior. Todo esto indica que la estructura de las cadenas de pensamiento producidas son muy sensibles a los ejemplos brindados en la instrucción de sistema. 

Imprimamos ahora los tiempos de ejecución:

In [29]:
print('Llama3.1:8b')
print_time_statistics(llama_report.invocations)
print("="*80)
print('Gemma3:4b')
print_time_statistics(gemma_report.invocations)

Llama3.1:8b
Total: 119.79620957374573
Average: 3.993206985791524
Max: 7.639201879501343
Min: 2.3192195892333984
Gemma3:4b
Total: 66.35315155982971
Average: 2.2117717186609904
Max: 2.7806663513183594
Min: 1.6886472702026367


De nuevo, Gemma3:4b tuvo los tiempos de ejecución más cortos. 

### 3.3 Variación de Tarea

Nuestro último experimento consiste en variar el problema aritmético a resolver por el LLM, dejando fijo la instrucción de sistema y el ejemplo de entrenamiento.

In [11]:
from typing import List, Dict, Optional

class TaskVariationExperiment(AbstractExperiment):
    """
    Orchestrates experiments of task variation
    keeping the instruction and shot constant. 
    """

    def __init__(
        self,
        formatter: PromptFormatter,
        parser: AnswerParser,
        instruction: SystemInstruction,
        shot: Shot,
        tasks: List[Shot]
    ):
        self._formatter = formatter
        self._parser = parser
        self._instruction = instruction
        self._shot = shot
        self._tasks = tasks
        
        self._system_prompt = self._formatter.format_system_prompt(instruction, shot)

    def execute(self, llm: LLMProtocol) -> ExperimentReport:
        
        invocations = []
        for task in self._tasks:
            
            user_prompt = self._formatter.format_user_prompt(task)
            invocation = llm.call(system=self._system_prompt, prompt=user_prompt)
            invocations.append(invocation)

        return self._fill_report(invocations)

    def _fill_report(self, invocations: LLMInvocationData) -> ExperimentReport: 
        """ Computes the main results of the experiment """
        
        report = ExperimentReport(invocations=invocations)
        
        correct_matches = 0
        for i in range(len(report.invocations)):
            invocation = report.invocations[i]
           
            try:
                reasoning, answer = self._parser.parse(invocation)

                if str(answer).strip() == str(self._tasks[i].answer).strip():
                    correct_matches += 1
                else:
                    report.incorrect_answers.append(invocation)

                if reasoning not in report.distinct_reasonings:
                    report.distinct_reasonings.append(reasoning)
            
            except ValueError:
                report.wrongly_formatted_invocations.append(invocation)

        if report.invocations:
            report.accuracy = correct_matches / len(report.invocations)

        return report

Ejecutamos el experimento.

In [12]:
experiment = TaskVariationExperiment(formatter, parser, instructions[0], shots[0], shots) 

gemma_report = experiment.execute(gemma)
llama_report = experiment.execute(llama)

print('gemma3:4b')
print(gemma_report.summary())
print('='*80)
print('llama3.1:8b')
print(llama_report.summary())

gemma3:4b
--- Experiment Summary ---
Total Invocations:  30
Success (Accuracy): 36.67%
Format Errors:      16
Incorrect Answers:  3
--------------------------
llama3.1:8b
--- Experiment Summary ---
Total Invocations:  30
Success (Accuracy): 63.33%
Format Errors:      1
Incorrect Answers:  10
--------------------------


¡Super interesante! El rendimiento es bajo pero no catastrófico. Analicemos primero los errores de Gemma3:4b, pero esta vez imprimiendo la instrucción de sistema así como la respuesta del LLM por cada invocación. Estos son los errores de formato:

In [13]:
for x in gemma_report.wrongly_formatted_invocations:
    print(f"User Prompt: {x.user_prompt}")
    print(f"LLM Answer:\n{x.raw_answer}")
    print("="*80)

User Prompt: Statement: If you buy 3 shirts for $15 each and pay with a $50 bill, how much change will you receive?
<Reasoning>

LLM Answer:
<Reasoning>
To calculate the total cost of the shirts, we multiply the number of shirts by the price per shirt: 3 shirts * $15/shirt = $45.
Then, we subtract the total cost from the amount paid to find the change: $50 - $45 = $5.
</Reasoning>
<Answer>
$5

User Prompt: Statement: 15% of a class of 40 students failed a test. How many students passed?
<Reasoning>

LLM Answer:
<Reasoning>
To find the number of students who failed, we need to calculate 15% of 40. This can be done by multiplying 0.15 by 40. Then, we subtract this number from the total number of students to find the number of students who passed.
</Reasoning>
<Answer>
40 - (0.15 * 40) = 40 - 6 = 34
34

User Prompt: Statement: If a recipe calls for 3/4 cup of sugar and you want to double the recipe, how much sugar do you need?
<Reasoning>

LLM Answer:
<Reasoning>
To double the recipe, we 

Observamos el mismo error de formato de olvidar de cerrar la respuesta final con la etiqueta `</Answer>`, pero si analizamos exhaustivamente las tareas veremos que el modelo genero respuestas correctas para cada una de ellas. 

Los errores de respuesta son los siguientes:

In [14]:
for x in gemma_report.incorrect_answers:
    print(f"User Prompt: {x.user_prompt}")
    print(f"LLM Answer:\n{x.raw_answer}")
    print("="*80)

User Prompt: Statement: A shirt is originally priced at $40 but is on sale for 25% off. What is the discount amount?
<Reasoning>

LLM Answer:
<Reasoning>
To calculate the discount amount, we need to find 25% of the original price ($40). We can do this by multiplying the original price by the discount percentage (expressed as a decimal). 25% is equivalent to 0.25.
</Reasoning>
<Answer>
$10
</Answer>
User Prompt: Statement: A car travels at a constant speed of 60 miles per hour. How far will it travel in 3.5 hours?
<Reasoning>

LLM Answer:
<Reasoning>
We can use the formula distance = speed × time.  We are given the speed (60 miles per hour) and the time (3.5 hours).  We need to calculate the distance.
</Reasoning>
<Answer>
210 miles
</Answer>
User Prompt: Statement: A crate of 24 sodas costs $18. How much does each soda cost?
<Reasoning>

LLM Answer:
<Reasoning>
To find the cost of each soda, we need to divide the total cost of the crate by the number of sodas in the crate. So, we will 

Las respuestas son correctas pero están en un formato que no esperábamos, por lo que no pudimos contarlas como tal. 

Imprimamos el error de formato de Llama3.1:8b:

In [15]:
print(llama_report.wrongly_formatted_invocations[0].raw_answer)

To find the total number of loaves made by the baker, we need to add the number of loaves made in the morning (12) to the number of loaves made in the afternoon (15). 

</Reasoning>

<Answer>
27
</Answer>


El modelo no escribió la etiqueta de aperatura `<Reasoning>`. 

Los errores de respuesta son los siguientes:

In [17]:
for x in llama_report.incorrect_answers:
    print(f"User Prompt:\n{x.user_prompt}")
    print(f"LLM Answer:\n{x.raw_answer}")
    print("="*80)

User Prompt:
Statement: If you buy 3 shirts for $15 each and pay with a $50 bill, how much change will you receive?
<Reasoning>

LLM Answer:
<Reasoning>
To find the total cost of the shirts, multiply the number of shirts by the price per shirt: 3 x $15 = $45.
Then, subtract the total cost from the amount paid with: $50 - $45 = $5.
</Reasoning>

<Answer>
$5
</Answer>
User Prompt:
Statement: A shirt is originally priced at $40 but is on sale for 25% off. What is the discount amount?
<Reasoning>

LLM Answer:
<Reasoning>
To find the discount amount, calculate 25% of the original price by multiplying it by 0.25 (since 25% = 25/100 = 0.25). Then subtract this value from the original price to get the discount amount.
</Reasoning>

<Answer>
$10
</Answer>
User Prompt:
Statement: If a rectangle has a length of 12cm and a width of 5cm, what is its perimeter?
<Reasoning>

LLM Answer:
<Reasoning>
To find the perimeter of a rectangle, we need to add up the lengths of all four sides. Since opposite s

¡Nuevas sorpresas! Si bien algunos errores de respuesta fueron debido a la adición de un símbolo especial como `$`, esta vez el modelo razonó incorrectamente alguno de los problemas dados. Esto nos da motivos para desconfiar de la capacidad de los LLMs pequeños para resolver tareas de razonamiento. Es interesante los giros de su cadena de pensamiento para el problema de las frutas:

* *If Maria has 3 apples and Julian has 1 banana and the double of apples than Maria, how many fruits do they have in total?*

Terminemos el análisis de este experimento imprimiendo los tiempos de ejecución:

In [21]:
print('Llama3.1:8b')
print_time_statistics(llama_report.invocations)
print("="*80)
print('Gemma3:4b')
print_time_statistics(gemma_report.invocations)

Llama3.1:8b
Total: 176.29619097709656
Average: 5.8765396992365515
Max: 17.20921564102173
Min: 2.571070671081543
Gemma3:4b
Total: 134.65008544921875
Average: 4.488336181640625
Max: 21.152971982955933
Min: 2.2634665966033936


## 4. Conclusiones

En esta notebook hemos estudiado la capacidad de LLMs pequeños como Llama3.1:8b y Gemma3:4b para resolver problemas aritméticos y lógicos simples y a estructurar sus respuestas. Construimos un lote de problemas y sus soluciones y lo utilizamos para evaluar las cadenas de pensamiento generados por los modelos al variar diferentes parámetros en su instrucción. Algunas observaciones de estos experimentos son:

* **Tiempo:** El modelo más pequeño Gemma3:4b produce respuestas más rápido que Llama3.1:8b.
* **Razonamientos:** Los ejemplos de entrenamiento en la instrucción de sistema afectan las cadenas de pensamiento generados por los modelos.
* **Robustez ante redacción:** Pequeñas variaciones en la redacción de la instrucción de sistema no afectan sustancialmente el comportamiento de los modelos.
* **Formato de respuesta:** Los LLMs producen sistemáticamente los mismos errores en los formatos de sus respuestas al olvidar una etiqueta de apertura o de cierre. Se necesita mayor trabajo para mejorar la fidelidad de los modelos a seguir el formato de respuesta solicitado. En flujos de trabajo futuros podrían crearse componentes normalizadoras o correctoras que iterativamente mejoran una respuesta.
* **Correctitud de razonamientos:** El modelo Gemma3:4b produjo respuestas correctas para todos los problemas, aunque estas no pudieron ser clasificadas como tal por la manera en la que fueron escritas. Por otro lado, Llama3:8b presentó errores en sus razonamientos que lo hicieron generar respuestas erróneas a los problemas planteados. 

## Referencias

* **Wei, J., et al. (2022).** *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models.* Advances in Neural Information Processing Systems (NeurIPS).

* **Touvron, H., et al. (2023).** *Llama 2: Open Foundation and Fine-Tuned Chat Models.* Meta AI Research.

* **Team, Gemma. (2024).** *Gemma: Open Models Based on Gemini Research and Technology.* Google DeepMind.

* **Wolf, T., et al. (2020).** *Transformers: State-of-the-Art Natural Language Processing.* Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing: System Demonstrations.